GeoPandas Practice Sheet 
1.	Complete the hierarchy:  

    1. Shapely geometry  
    2. GeoSeries (a column of Shapely geometries, plus a CRS)
    3. GeoDataFrame 

2.	What type of object is returned by each expression? Write Shapely geometry, GeoSeries, GeoDataFrame, or pandas Series. 
parks.geometry -> GeoSeries
parks.geometry.iloc[0] -> Shapely Geometry
parks.iloc[0] parks.iloc[[0]] -> GeoDataFrame

3.	Explain in one sentence why `parks.geometry.iloc[0]` does not behave exactly like `parks.geometry`. 
Parks.geometry.iloc[0] -> a single bare shapely object, not a GeoSeries, so it has not CRS, no index, and none of the vectorized GeoSeries/panda methods.

4.	Write code to display all column names in `parks` as a regular Python list. 
Parks.columns.tolist()

5.	Write code to select only the `NAME` and `geometry` columns from `parks`. 
Parks[[“NAME”, “geometry”]]

6.	Write code to select the first three rows of `parks` while keeping the result as a GeoDataFrame. 
Parks.iloc[:3]

7.	Write code to inspect the geometry type of every feature in `parks`. 
Parks.geom_type
Parks.geom_type.value_counts()

8.	What does `parks.crs` tell you? 
Parks.crs gives coordinate reference system. Shows how coordinate numbers map to locations on earth, whether the CRS is geographic or projected, and the units the coordinates and any distance or area calculations will be in.

9.	Why can one point still produce a complete graph when you call `.plot()`? 
Since plot() builds a full Matplotlib figure with axes, ticks, and frame. The axes auto-scale around the data; so the single point will show up as a marker inside of a complete plot.

10.	Assume `one_park = parks.geometry.iloc[0]`. What library provides the geometry object stored in `one_park`? 
One_park is a shapely polygon/multipolygon.
stops.crs # EPSG:4326 

11.	Would EPSG:4326 be a good CRS for measuring distance in feet? Explain briefly. 
No because EPSG4326 will store latitude and longitude in degrees, so the distance() would return a degree value.

12.	Write code to create `stops_nc` by reprojecting `stops` to EPSG:2264. 
Stops_nc = stops.to_crs(epsg=2264)

13.	Write code to create an `area_sqft` column in `parks_nc` using polygon geometry area. 
Parks_nc[“area_sqft”] = parks_nc.geometry.area

14.	Write code to calculate the distance from every stop in `stops_nc` to a single polygon named `park_1`. 
Park_1 = parks_nc.geometry.iloc[0]
Stops_nc[“dist_to_park1”] = stops_nc.distance(park_1)

15.	If `park_1` is a polygon, does `.distance(park_1)` measure distance to the polygon centroid? Explain. 
No, because .distance() returns the shortest distance to the nearest point on the polygon boundary. It does not measure to the centroid.

16.	Explain each 
Predicate	Meaning
within	A will lie entirely inside B. No part of A is outside of B.
contains	B lies entirely inside A (reverse of within)
intersects	A and B share at least one point (overlap, touch, or one inside the other)
touches	A and B share boundary points only; interiors do not overlap.

17.	In `gpd.sjoin(parks, urban, predicate="within")`, which geometry is being tested as within which? 
Left layer is tested against the right layer, so each park is tested for being within an urban polygon.

18.	A park crosses an urban-area boundary. Would `within` be True? Would `intersects` be True? 
Within is False, because part of the park lies outside of the urban area. Intersects is True, because the two still share area.

19.	Write code that returns how many stops are inside each polygon in a gpd named `parks` that has three polygons. 
Joined = gpd.sjoin(stops, parks, how=”inner”, predicate=“within”)
Counts = joined.groupby(“index_right”).size()
Parks[“n_stops”] = counts.reindex(parks.index, fill_value=0)
urban_parks = gpd.sjoin(parks, urban, predicate="within") 

20.	What is the purpose of `gpd.sjoin()`? 
It combines the attributes of two GeoDataFrames based on spatial relationships between their geometries, such as within or intersects. It does not use a shared key. 

21.	Why is a spatial join different from a regular pandas join on a common ID field? 
A pandas join matches rows on equal values in a common column. A spatial join matches rows on where the geometries are located relative to each other so it doesn’t need a shared ID.

22.	Write a spatial join that keeps bus stops that fall inside park polygons. Use `stops` as the left GeoDataFrame and `parks` as the right GeoDataFrame. 
Stops_in_parks = gpd.sjoin(stops, parks, how=”inner”, predicate=”within”)

23.	After a point-in-polygon spatial join, how could you count how many bus stops were matched to each park? Describe the idea or write code. 
Stops_in_parks.groupby(“index_right”).size()
Stops_in_parks.groupby(“NAME”).size()

24.	Why can a polygon-to-polygon `intersects` spatial join create multiple output rows if each original layer was one polygon in size? 
A spatial join outputs one row per matching pair, not one row per original feature. If a left polygon intersects more than one right polygon, it will appear once for each match, so the output can have more rows than either input.

25.	You want to answer: “Which parks are within 500 feet of a bus stop?” Write how would you do this with 
GeoPandas  
a.	using a distance and apply method.  
	Parks_nc[“min_stop_dist”] = parks_nc.geometry.apply(
    Lambda park: stops_nc.distance(park).min()
    )
     Parks_near_stop = parks_nc[parks_nc[“min_stop_dist”] <= 500]
b.	using buffers 
	stops_buf = stops_nc.copy()
    stops_buf[“geometry”] = stops_nc.buffer(500)

matches = gpd.sjoin(parks_nc, stops_buf, how=”inner”, predicate=”intersects”)

parks_near_stop = parks_nc[parks_nc.index.isin(matches.index)]
